In [1]:
import numpy as np

In [ ]:
import numpy as np

class RNN:  # en una rnn los datos son (T, B, D)
    def __init__(self, input_size, hidden_size, output_size, layers, activation, batch_size):
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.output_size = output_size
        self.layers = layers
        self.activation = activation
        self.batch_size = batch_size

        # Pesos
        self.pesos_entrada = np.random.randn(input_size, hidden_size) * 0.01      # (D, H)
        self.pesos_hidden  = np.random.randn(layers, hidden_size, hidden_size) * 0.01  # (L, H, H) recurrentes
        self.pesos_vertical = np.random.randn(layers, hidden_size, hidden_size) * 0.01 # (L, H, H) verticales
        self.pesos_salida  = np.random.randn(hidden_size, output_size) * 0.01     # (H, O)

        # Bias
        self.bias_hidden = np.zeros((1, hidden_size))   # compartido para todas las capas
        self.bias_salida = np.zeros((1, output_size))

        # Memorias para backward
        self.salidas_intermedias = {}  # (t, L) -> h_t^L
        self.inputs = None
        self.salida_final = []

    def forward(self, datos):
        """
        datos: (T, B, D)
        """
        T, B, D = datos.shape
        H = self.hidden_size
        L_count = self.layers

        # h[L, B, H]: estado oculto por capa
        h = np.zeros((L_count, B, H))
        self.inputs = np.zeros((T, B, D))
        self.salidas_intermedias = {}
        self.salida_final = []

        for t in range(T):
            x_t = datos[t]         # (B, D)
            self.inputs[t] = x_t

            for L in range(L_count):
                h_anterior_t = h[L]            # h_{t-1}^L
                W_hh = self.pesos_hidden[L]    # (H, H)

                if L == 0:
                    # capa 0: entrada externa + recurrencia
                    linear = x_t @ self.pesos_entrada + h_anterior_t @ W_hh + self.bias_hidden
                else:
                    # capas superiores: vertical + recurrencia
                    h_prev_layer = self.salidas_intermedias[(t, L-1)]  # h_t^{L-1}
                    W_vert = self.pesos_vertical[L-1]
                    linear = h_prev_layer @ W_vert + h_anterior_t @ W_hh + self.bias_hidden

                h_t = np.tanh(linear)
                h[L] = h_t
                self.salidas_intermedias[(t, L)] = h_t

            # salida solo desde la última capa
            o_t = h[L_count-1] @ self.pesos_salida + self.bias_salida  # (B, O)
            self.salida_final.append(o_t)

        return np.stack(self.salida_final, axis=0), h   # (T,B,O), (L,B,H)

    def backward(self, learning_rate, error):
        """
        error: dL/dO_t con forma (T, B, O)
        """
        T, B, O = error.shape
        H = self.hidden_size
        L_count = self.layers
        L_final = L_count - 1

        # Gradientes acumulados
        dWhy_total   = np.zeros_like(self.pesos_salida)     # (H, O)
        dBy_total    = np.zeros_like(self.bias_salida)      # (1, O)
        dWxh_total   = np.zeros_like(self.pesos_entrada)    # (D, H)
        dWhh_total   = np.zeros_like(self.pesos_hidden)     # (L, H, H)
        dWvert_total = np.zeros_like(self.pesos_vertical)   # (L, H, H)
        dBh_total    = np.zeros_like(self.bias_hidden)      # (1, H) bias compartido

        # d_h_next_time[L, B, H]: error recurrente que viene de t+1
        d_h_next_time = np.zeros((L_count, B, H))

        for t in reversed(range(T)):
            dO_t = error[t]  # (B, O)
            h_t_final = self.salidas_intermedias[(t, L_final)]  # (B, H)

            # grad salida -> última capa oculta
            dWhy_total += h_t_final.T @ dO_t         # (H, O)
            dBy_total  += np.sum(dO_t, axis=0, keepdims=True)  # (1, O)

            # error que entra por la última capa desde la salida
            d_h = np.zeros((L_count, B, H))
            d_h[L_final] += dO_t @ self.pesos_salida.T   # (B, H)

            # sumamos también lo que viene del futuro (t+1)
            d_h += d_h_next_time

            # error vertical que baja de la capa superior a la inferior
            d_h_prev_vert = np.zeros((B, H))

            for L in reversed(range(L_count)):
                h_t_L = self.salidas_intermedias[(t, L)]   # (B, H)

                # error total que entra al tanh en esta capa
                dH_total = d_h[L] + d_h_prev_vert

                # derivada de tanh: dS = dH_total * (1 - h^2)
                dS_t = dH_total * (1.0 - h_t_L**2)   # (B, H)

                # acumula bias oculto (compartido entre capas)
                dBh_total += np.sum(dS_t, axis=0, keepdims=True)  # (1, H)

                # recurrente: W_hh[L] y contribución al tiempo t-1
                h_prev_time = self.salidas_intermedias.get((t-1, L), np.zeros((B, H)))
                dWhh_total[L] += h_prev_time.T @ dS_t           # (H, H)
                d_h_next_time[L] = dS_t @ self.pesos_hidden[L].T  # (B, H) para t-1

                if L == 0:
                    # entrada externa
                    x_t = self.inputs[t]   # (B, D)
                    dWxh_total += x_t.T @ dS_t   # (D, H)
                    d_h_prev_vert = np.zeros((B, H))  # no hay capa inferior
                else:
                    # conexión vertical desde capa L-1
                    h_prev_vert = self.salidas_intermedias[(t, L-1)]   # (B, H)
                    dWvert_total[L-1] += h_prev_vert.T @ dS_t          # (H, H)
                    d_h_prev_vert = dS_t @ self.pesos_vertical[L-1].T  # error que baja a L-1

        # Actualización de pesos
        self.pesos_salida   -= learning_rate * dWhy_total
        self.bias_salida    -= learning_rate * dBy_total
        self.pesos_hidden   -= learning_rate * dWhh_total
        self.pesos_vertical -= learning_rate * dWvert_total
        self.pesos_entrada  -= learning_rate * dWxh_total
        self.bias_hidden    -= learning_rate * dBh_total

        # Devuelvo gradientes por si quieres debuggear numéricamente
        return {
            "dWhy": dWhy_total,
            "dBy": dBy_total,
            "dWxh": dWxh_total,
            "dWhh": dWhh_total,
            "dWvert": dWvert_total,
            "dBh": dBh_total
        }
